# Silence measurements of every file, compared with the original tables (gate A5)

Run at a fixed commit on a Colab runtime, after
`notebooks/02_migrate_wavlm_features.ipynb` has passed. No GPU is needed and
no encoder is loaded: the notebook reads the audio only. The measurements do
not depend on the encoder, so the notebook runs once.

Every file of the four sets (ASVspoof 2019 LA train, dev and eval, and
In-the-Wild; 153,240 files) is loaded once and measured three ways
(`silence.measure_items`): the silence of the untrimmed audio, the silence
that remains after trimming at 40 dB, and what the trimming rule removes at
40, 20 and 15 dB. The chunks of the measurement are written to the local disk
of the runtime only. Drive receives the finished tables: one per-file table
per set and the summary tables under `measurements/`, the scores of the
silence-only classifiers under `runs/baselines/scores/`, one figure, and the
check tables under `checks/`.

## Gate A5

Gate A5 is not part of the pre-registered analysis plan. Its definition is
written in `docs/decisions.md` (Section 1), before any run.
The gate passes when every check below passes.

1. **Per file.** Every column of the three original per-file tables (the
   untrimmed audio, the audio trimmed at 40 dB, the removed amounts), for
   every file of the four sets, on `audio_id`
   (`silence.compare_with_original_files`). The tolerance is 1e-9
   (`verify.SECONDS_TOLERANCE`): a count must be equal; seconds may differ
   in the last digit, which the original code lost when it read its tables
   with the default parser.
2. **The seven original tables** of the mapping file (`silence_summary`,
   `residual_silence_summary`, `trim_amount`, `padding_only_baseline`,
   `padding_by_label`, `silence_only_baseline`,
   `residual_silence_baseline`), every column that is not a key, with the
   tolerance of the mapping file, which is 1e-9 for these tables
   (`verify.compare_with_legacy`). A row that only one table has fails.
3. **The stored features.** For every file and each of 40, 20 and 15 dB, the
   measured length after trimming, cut to the window, equals the length
   stored in the shards of WavLM Base; the fallback and too-short counts per
   set and class are equal (`silence.compare_with_shards`).
4. **In-the-Wild.** Every file was recorded at 16 kHz; 19,387 files are at
   most 4 s long and 12,392 are longer (`silence.duration_checks`).
5. **The printed numbers of Section 4.7** on the leading silence that
   remains after trimming at 40 dB, each rounded once to the digits the
   manuscript prints (`intext.compare_stated`): the smallest and the largest
   share of files with more than 50 ms over the four sets, in the whole file
   and inside the 4-second window; the share on the evaluation set by class;
   the smallest and the largest median of the remaining length. Every file
   is counted, also a file that the fallback rule kept untrimmed
   (`silence.remaining_silence`). The numbers of the manuscript are read
   from the in-text registry (`intext.STATED`); none is typed into this
   notebook.
6. **The quiet In-the-Wild files.** The number of In-the-Wild files whose
   loudest frame has an RMS below `silence.untrimmable_below(40)`, and
   their class, are those the manuscript states (`intext.STATED_QUIET`,
   `intext.compare_quiet`). The files are listed.

No tolerance is loosened: a comparison that fails at 1e-9 stops the notebook
in the cell "Gate A5", after every table and every check table has been
written, so that the difference can be investigated.

Reported and not part of the gate: the quiet files of the other sets and
thresholds, the quartiles of the remaining length per set and class, the
counts of the original feasibility table next to the measured counts, and
the figure of the distributions.

The steps:

1. what this run measures; the original files and tables;
2. inputs: the two datasets on the local disk and their file lists;
3. the timed check of 100 files per set and the estimate;
4. the measurement of every file; the per-file tables are written to Drive;
5. the per-file comparison with the original tables;
6. the stored lengths and the fallback counts of the shards;
7. the sampling rate and the duration subsets of In-the-Wild;
8. the seven tables and their comparison with the original tables;
9. the silence that remains after trimming and the printed numbers;
10. the quiet files;
11. the figure;
12. gate A5 (a hard stop).

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Google Drive is not mounted"

### Repository at a fixed commit

Set `COMMIT` to the commit of this repository that is being checked: the
full hash or at least its first 7 characters. The repository is public and
is cloned without a token. If the Colab secret `GITHUB_TOKEN` exists and
this notebook has access to it, the clone uses it (a repository that is not
public needs one); the token is not kept in the clone and is replaced by
`***` in everything git prints.

In [ ]:
import subprocess

REPOSITORY = "Nooralabd/silence-shortcut-wavlm-layers"
COMMIT = ""          # the commit to run: the full hash or at least its first 7 characters
WORKDIR = "/content/silence-shortcut-wavlm-layers"

assert len(COMMIT) >= 7 and set(COMMIT) <= set("0123456789abcdef"), \
    "set COMMIT to the commit that is being checked (full hash or at least its first 7 characters)"
try:
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:    # the secret does not exist, or this notebook has no access to it
    token = ""       # a public repository is cloned without a token


def git(*arguments):
    """Run git and print what it prints; the token is never printed."""
    result = subprocess.run(["git", *arguments], capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if token:
        output = output.replace(token, "***")
    if output:
        print(output)
    assert result.returncode == 0, "git failed: see the lines above"
    return output


def clone(repository, folder):
    """Clone a repository, or fetch into an existing clone; a token, if there is one, is not kept in it."""
    address = f"https://{token}@github.com/{repository}.git" if token else f"https://github.com/{repository}.git"
    if not os.path.isdir(folder):
        git("clone", "-q", address, folder)
        git("-C", folder, "remote", "set-url", "origin", f"https://github.com/{repository}.git")
    else:
        git("-C", folder, "fetch", "-q", address, "+refs/heads/*:refs/remotes/origin/*")


clone(REPOSITORY, WORKDIR)
git("-C", WORKDIR, "checkout", "-q", COMMIT)
git("-C", WORKDIR, "rev-parse", "HEAD")
%cd {WORKDIR}

In [ ]:
# Installation. torch, torchaudio and torchcodec are the builds that extracted
# the original features: the pinned versions for CUDA 13.0. If anything pinned
# is missing or has another version, everything is installed in one pass and
# the cell always stops for a restart: a kernel never continues after pip has
# run in it (the runtime has imported numpy before this cell, and a module
# that is already loaded stays loaded when pip replaces it on disk).
# On the run after the restart nothing is installed: the cell checks the
# installed versions and the versions of the modules that the kernel loads.
import importlib
import importlib.metadata
import re
import sys

TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
TORCH_FAMILY = ("torch", "torchaudio", "torchcodec")          # installed from TORCH_INDEX
INSTALLED_ONCE = os.path.join(os.path.dirname(WORKDIR), "installed_once.txt")
OTHERS = os.path.join(os.path.dirname(WORKDIR), "requirements_without_torch.txt")
CONSTRAINT = os.path.join(os.path.dirname(WORKDIR), "torch_constraint.txt")

with open(os.path.join(WORKDIR, "requirements.txt"), encoding="utf-8") as fh:
    requirements = [line.strip() for line in fh if line.strip() and not line.lstrip().startswith("#")]
pins = dict(line.split("==") for line in requirements if "==" in line)
BUILDS = {name: pins[name] + "+" + TORCH_INDEX.rsplit("/", 1)[1] for name in TORCH_FAMILY}


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def not_as_pinned():
    """{package: (installed, wanted)} for everything that is missing or has another version."""
    found = {}
    for line in requirements:
        name = re.split(r"[=<>!~ ]", line, maxsplit=1)[0]
        have, want = installed(name), BUILDS.get(name, pins.get(name))
        same = have is not None and (want is None or (have if name in BUILDS else have.split("+")[0]) == want)
        if not same:
            found[name] = (have, want)
    if installed("ssl-silence") is None:
        found["ssl-silence"] = (None, "this repository")
    return found


different = not_as_pinned()
if different:
    assert not os.path.exists(INSTALLED_ONCE), (
        f"the libraries were installed once already and these are still not as pinned: {different}. "
        "Stop: do not restart again")
    imported = sorted(name for name in (*TORCH_FAMILY, "transformers", "ssl_silence") if name in sys.modules)
    assert not imported, (f"{imported} were imported before the installation: "
                          "Runtime > Restart session, then run the notebook again from the top")
    with open(OTHERS, "w", encoding="utf-8") as fh:
        for line in requirements:
            if re.split(r"[=<>!~ ]", line, maxsplit=1)[0] not in TORCH_FAMILY:
                print(line, file=fh)
    with open(CONSTRAINT, "w", encoding="utf-8") as fh:
        for name in TORCH_FAMILY:
            print(f"{name}=={BUILDS[name]}", file=fh)
    # The build is part of the pin (torch==2.11.0+cu130): for a pin without it pip
    # keeps another build of the same version that the runtime ships (+cpu).
    FAMILY_PINS = " ".join(f"{name}=={BUILDS[name]}" for name in TORCH_FAMILY)
    # torchvision of the runtime is built for its own torch and breaks imports
    # with another one; this package does not use it.
    !pip uninstall -q -y torchvision
    !pip install -q {FAMILY_PINS} --index-url {TORCH_INDEX}
    !pip install -q -r {OTHERS} -c {CONSTRAINT}
    !pip install -q --no-deps -e .
    importlib.invalidate_caches()
    # Before the restart: everything on disk is as pinned, so the next run installs nothing.
    different = not_as_pinned()
    assert not different, f"pip did not install these as pinned: {different}: stop"
    with open(INSTALLED_ONCE, "w", encoding="utf-8") as fh:
        print("the pinned libraries were installed in this runtime", file=fh)
    # pip has run in this kernel: it always stops here.
    raise RuntimeError("The pinned libraries were installed. "
                       "Runtime > Restart session, then run the notebook again from the top")

# Nothing was installed in this run. The modules that this kernel loads must
# be the pinned versions too, not only the files on disk.
LOADED = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "scikit-learn": "sklearn",
          "torch": "torch", "torchaudio": "torchaudio", "transformers": "transformers"}
for name, module in LOADED.items():
    loaded = importlib.import_module(module).__version__
    assert loaded.split("+")[0] == pins[name], (
        f"{module} {loaded} is loaded in this kernel and the pin is {pins[name]}: it was imported before pip "
        "replaced it. Runtime > Restart session, then run the notebook again from the top")

import torch
import torchaudio
import transformers

print("loaded:", {name: sys.modules[module].__version__ for name, module in LOADED.items()})
print("torch", torch.__version__, "| CUDA of the build:", torch.version.cuda)
print("torchaudio", torchaudio.__version__, "| torchcodec", installed("torchcodec"),
      "| transformers", transformers.__version__)
if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))   # a running kernel does not see a new editable install

In [ ]:
import subprocess
import sys

import pandas as pd

from ssl_silence import config as cfg
from ssl_silence import provenance, verify

pd.set_option("display.max_rows", 200, "display.width", 200)
HEAD = provenance.git_commit()     # the full hash, with "-dirty" if a tracked file was changed
assert HEAD.startswith(COMMIT) and not HEAD.endswith("-dirty"), (HEAD, COMMIT)
print("commit:", HEAD)
print("root:", cfg.ROOT)

In [ ]:
# The quota of the Google account is read from the Drive API (this asks for
# access to the account); the mounted folder does not report it.
QUOTA = cfg.drive_quota()
print("Google Drive: {usage_gb:.1f} GB used of {limit_gb:.1f} GB, {free_gb:.1f} GB free".format(**QUOTA),
      "(GB = 10^9 bytes; required: %d GB free)" % cfg.MIN_FREE_GB)
cfg.require_free_space(quota=QUOTA)
print("local disk of the runtime: %.1f GB free" % cfg.local_free_space_gb())

### Mapping file

`legacy_map.yaml` names the original folders, files and labels. It is kept on
Drive only, at `<root>/migration/legacy_map.yaml`, and must have no empty
value (`verify.load_map` stops otherwise).

In [ ]:
MAP_PATH = os.path.join(cfg.migration_dir(), "legacy_map.yaml")
assert os.path.exists(MAP_PATH), f"upload legacy_map.yaml to {MAP_PATH}"
legacy = verify.load_map(MAP_PATH)
FORBIDDEN = legacy["forbidden"]
CHECKS = cfg.checks_dir()
checks = {}


def record(name, table, stop=True):
    """Write a check table to Drive, show it, and stop if a row fails."""
    path = verify.write_check(table, name, CHECKS, FORBIDDEN)
    checks[name] = table
    display(table)
    if stop and "ok" in table.columns:
        verify.require_ok(table, name)
    return path

### What this run measures

Nothing is set here. The chunks of the measurement stay on the local disk of
the runtime; the original per-file tables and the seven original tables are
read from their migrated copies, whose sha256 is recorded.

In [ ]:
from ssl_silence import datasets, plots, shards, silence
from ssl_silence.paper import intext

SPLITS = cfg.SPLITS
SHARDS_OF = "wavlm-base"                                 # the shards whose stored lengths are compared
CONDITIONS = ("trim40", "trim20", "trim15")              # the trimmed conditions of the shards
ORIGINAL = os.path.join(cfg.migration_dir(), "original")
OUT = cfg.measurements_dir()                             # Drive: the finished tables
LOCAL = cfg.LOCAL_MEASUREMENTS                           # local disk of the runtime: the chunks
SCORES = os.path.join(cfg.baselines_dir(), "scores")     # Drive: the scores of the silence-only classifiers
assert not cfg.on_drive(LOCAL), f"the chunks of the measurement would be written to Drive: {LOCAL}"

original_paths = [os.path.join(ORIGINAL, "files", folder, f"{split}.csv")
                  for folder in silence.ORIGINAL_PARTS for split in SPLITS] \
    + [os.path.join(ORIGINAL, "tables", f"{name}.csv") for name in silence.GATE_TABLES]
missing = [path for path in original_paths if not os.path.exists(path)]
assert not missing, f"run notebooks/02_migrate_wavlm_features.ipynb first: {missing} are missing"
record("silence_original_files", pd.DataFrame({
    "file": [os.path.relpath(path, ORIGINAL).replace(os.sep, "/") for path in original_paths],
    "sha256": [provenance.sha256_file(path) for path in original_paths]}), stop=False)
print("chunks (local disk):", LOCAL, "| finished tables (Drive):", OUT)

### Inputs

Both archives are unpacked to the local disk of the runtime. The file lists
must be the complete sets.

In [ ]:
datasets.ensure_asvspoof(cfg.la_zip())
datasets.ensure_itw(cfg.itw_zip(), sha256=cfg.ITW_ZIP_SHA256)
file_lists = {split: datasets.items(split) for split in SPLITS}
for split, file_list in file_lists.items():
    print(split, datasets.check_items(file_list, split))

### Timed check

The first 100 files of every set are measured and the time is scaled to the
whole set. These measurements are discarded.

In [ ]:
import time

rows = []
for split in SPLITS:
    timed = file_lists[split].head(silence.TIMING_FILES)
    start = time.perf_counter()
    for path in timed["path"]:
        silence.measure_file(path)
    rows.append({"split": split, "files_timed": len(timed), "seconds": time.perf_counter() - start,
                 "files": len(file_lists[split])})
timing = pd.DataFrame(rows)
timing["estimated_minutes"] = timing["seconds"] / timing["files_timed"] * timing["files"] / 60
record("silence_timing", timing, stop=False)
print("estimate of the measurement:", round(float(timing["estimated_minutes"].sum()), 1), "minutes |",
      dict(zip(timing["split"], timing["estimated_minutes"].round(1))))

### Measurement

Every file, in chunks of 2,000 files on the local disk; a finished chunk is
not measured again while the runtime lives. The finished table of each set
is written to Drive (`measurements/<set>.csv`, with its meta file).

In [ ]:
wide = {}
for split in SPLITS:
    wide[split] = silence.measure_items(file_lists[split], LOCAL, split)
    provenance.write_table(wide[split], os.path.join(OUT, f"{split}.csv"))
raw = {split: silence.silence_table(table) for split, table in wide.items()}
residual = {split: silence.residual_table(table) for split, table in wide.items()}
print("files measured:", {split: len(table) for split, table in wide.items()})

### Per-file comparison with the original tables

Gate A5, point 1. One row per compared column, per part and set.

In [ ]:
per_file = pd.concat([silence.compare_with_original_files(wide[split], split, os.path.join(ORIGINAL, "files"))
                      for split in SPLITS], ignore_index=True)
record("silence_per_file", per_file, stop=False)
print("columns compared file by file:", len(per_file), "| failed:", int((~per_file["ok"]).sum()),
      "| largest difference:", per_file["max_abs_diff"].max())

### The stored lengths and the fallback counts of the shards

Gate A5, point 3. The shards of WavLM Base at 40, 20 and 15 dB store, for
every file, the length after trimming and after the centre crop, and whether
the fallback rule applied. Only these values are read from the shards.

In [ ]:
long = silence.removed_long(pd.concat([silence.removed_table(wide[split]) for split in SPLITS],
                                      ignore_index=True))
counts, same = [], []
for condition in CONDITIONS:
    lengths = shards.shard_lengths(cfg.feature_dir(SHARDS_OF, "unpadded", condition))
    counted, compared = silence.compare_with_shards(long, lengths, cfg.trim_top_db(condition))
    counts.append(counted.assign(condition=condition))
    same.append(compared.assign(condition=condition))
record("silence_fallback_counts", pd.concat(counts, ignore_index=True), stop=False)
record("silence_trim_lengths", pd.concat(same, ignore_index=True), stop=False)

### In-the-Wild: sampling rate and duration subsets

Gate A5, point 4.

In [ ]:
record("silence_itw_durations", silence.duration_checks(raw["itw"]), stop=False)

### The seven tables

Gate A5, point 2. The five summary tables (`silence.summary_tables`) and the
two tables of the silence-only classifiers, at the thresholds of the
original tables (`silence.BASELINE_TOP_DBS`). Each table is written to Drive
and compared with its original table.

In [ ]:
tables = silence.summary_tables(wide)
for name, condition in silence.BASELINE_TABLES.items():
    measured = raw if condition == silence.BASE_CONDITION else residual
    tables[name] = silence.silence_only_baselines(measured, silence.BASELINE_TOP_DBS[condition], condition,
                                                  SCORES, overwrite=True)
assert tuple(tables) == silence.GATE_TABLES
for name, table in tables.items():
    provenance.write_table(table, os.path.join(OUT, f"{name}.csv"))
    record(f"table_{name}", verify.compare_with_legacy(table, name, legacy, ORIGINAL), stop=False)

### The silence that remains after trimming at 40 dB

Gate A5, point 5. `silence.remaining_silence` counts, per set and class, the
files with more than 50 ms of leading silence in the audio trimmed at 40 dB,
in the whole file and inside the window, and gives the quartiles of the
remaining length. The numbers of the two entries of Section 4.7 are computed
from this table and compared with the numbers the manuscript prints.

Not part of the gate: the counts of the original feasibility table, shown
next to the measured counts of every file and of the files without fallback.

In [ ]:
share = silence.remaining_silence(residual)
provenance.write_table(share, os.path.join(OUT, os.path.basename(intext.RESIDUAL_SHARE)))
display(share)
stated = intext.compare_stated(intext.Tables.of({intext.RESIDUAL_SHARE: share}),
                               intext.remaining_silence_entries())
record("silence_printed_numbers", stated, stop=False)
record("silence_earlier_counts", silence.earlier_counts_report(share), stop=False)

### Quiet files

Gate A5, point 6. A file whose loudest frame has an RMS below
`silence.untrimmable_below(d)` cannot be trimmed at `d` dB. The counts per
set and class at 40, 20 and 15 dB and the files at 40 dB are written and
shown; the In-the-Wild files at 40 dB are compared with what the manuscript
states.

In [ ]:
quiet_counts = silence.quiet_file_counts(raw)
quiet = silence.quiet_files(raw)
provenance.write_table(quiet_counts, os.path.join(OUT, "quiet_file_counts.csv"))
provenance.write_table(quiet, os.path.join(OUT, "quiet_files.csv"))
display(quiet_counts)
display(quiet)
quiet_check = intext.compare_quiet(quiet)
record("silence_quiet_files", quiet_check, stop=False)

### Figure

The distribution of the leading silence of the untrimmed audio at 40 dB,
bonafide against spoof, one panel per set. Not part of the gate; the first
author compares it with the published figure.

In [ ]:
figure = plots.silence_ecdf_figure({plots.SPLIT_PANELS[split]: raw[split] for split in SPLITS})
print(plots.save_figure(figure, os.path.join(cfg.figures_dir(), "silence_ecdf"), print_width=False))

### Gate A5

Every check table of the gate has been written. The first one with a failed
row stops the notebook here.

In [ ]:
gate = {name: checks[name] for name in silence.GATE_CHECKS}
display(verify.summarise_checks(gate))
for name, table in gate.items():
    verify.require_ok(table, name)
print("gate A5 passed | files measured:", sum(len(table) for table in wide.values()),
      "| columns compared file by file:", len(per_file),
      "| original tables reproduced:", len(tables),
      "| printed numbers reproduced:", len(stated),
      "| quiet In-the-Wild files:", int(quiet_check["n_files"].iloc[0]))
print("tables written to", OUT, "| check tables written to", CHECKS)